Post-level data aggregating to User-level

In [ ]:
import json
from collections import defaultdict
from numbers import Number

# Input post-level JSON file containing extracted features
INPUT_PATH = "normal_with_features.json"

# Output user-level JSON file after aggregation
OUTPUT_PATH = "normal_user_level.json"

# Load post-level data
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

# Store intermediate aggregation results for each author
author_data = {}

for post in data:
    author = post.get("author")
    if not author:
        continue

    # Initialize the author entry the first time this author is seen
    if author not in author_data:
        author_data[author] = {
            "author": author,
            "post_count": 0,
            "feature_sums": defaultdict(float),
            "feature_counts": defaultdict(int),
            "subreddits": set(),
            "ids": [],
            "created_utcs": [],
            "raw_word_count_sum": 0.0,
            "raw_word_count_count": 0,
        }

    entry = author_data[author]
    entry["post_count"] += 1

    # Collect subreddit membership information
    if post.get("subreddit"):
        entry["subreddits"].add(post["subreddit"])

    # Collect post IDs if available
    if post.get("id"):
        entry["ids"].append(post["id"])

    # Collect timestamps if available
    if post.get("created_utc") is not None:
        entry["created_utcs"].append(post["created_utc"])

    # Aggregate post-level word counts if numeric
    if isinstance(post.get("word_count"), Number):
        entry["raw_word_count_sum"] += post["word_count"]
        entry["raw_word_count_count"] += 1

    # Aggregate numeric feature values across posts
    features = post.get("features", {})
    if isinstance(features, dict):
        for feat_name, feat_value in features.items():
            if isinstance(feat_value, Number):
                entry["feature_sums"][feat_name] += feat_value
                entry["feature_counts"][feat_name] += 1

# Final user-level output list
user_level_data = []

for author, entry in author_data.items():
    # Compute the mean value of each numeric feature for this user
    avg_features = {}
    for feat_name in entry["feature_sums"]:
        count = entry["feature_counts"][feat_name]
        if count > 0:
            avg_features[feat_name] = entry["feature_sums"][feat_name] / count

    # Build the final user-level record
    user_record = {
        "author": author,
        "post_count": entry["post_count"],
        "subreddits": sorted(entry["subreddits"]),
        "features": avg_features,
    }

    # Add the first and last observed timestamps if available
    if entry["created_utcs"]:
        user_record["first_created_utc"] = min(entry["created_utcs"])
        user_record["last_created_utc"] = max(entry["created_utcs"])

    # Add mean raw word count if available
    if entry["raw_word_count_count"] > 0:
        user_record["mean_raw_word_count"] = (
            entry["raw_word_count_sum"] / entry["raw_word_count_count"]
        )

    user_level_data.append(user_record)

# Save aggregated user-level data
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(user_level_data, f, ensure_ascii=False, indent=2)

print(f"Original number of posts: {len(data)}")
print(f"Number of users: {len(user_level_data)}")
print(f"Saved to: {OUTPUT_PATH}")

In [ ]:
import json
from collections import defaultdict
from numbers import Number

# Input post-level JSON file containing extracted features
INPUT_PATH = "depression_with_features.json"

# Output user-level JSON file after aggregation
OUTPUT_PATH = "depression_user_level.json"

# Load post-level data
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

# Aggregation container for each author
author_data = {}

for post in data:
    author = post.get("author")
    if not author:
        continue

    if author not in author_data:
        author_data[author] = {
            "author": author,
            "post_count": 0,
            "feature_sums": defaultdict(float),
            "feature_counts": defaultdict(int),
            "subreddits": set(),
            "ids": [],
            "created_utcs": [],
            "raw_word_count_sum": 0.0,
            "raw_word_count_count": 0,
        }

    entry = author_data[author]
    entry["post_count"] += 1

    # Retain selected metadata fields if available
    if post.get("subreddit"):
        entry["subreddits"].add(post["subreddit"])
    if post.get("id"):
        entry["ids"].append(post["id"])
    if post.get("created_utc") is not None:
        entry["created_utcs"].append(post["created_utc"])

    # Aggregate top-level word_count if available
    if isinstance(post.get("word_count"), Number):
        entry["raw_word_count_sum"] += post["word_count"]
        entry["raw_word_count_count"] += 1

    # Aggregate all numeric features inside the "features" dictionary
    features = post.get("features", {})
    if isinstance(features, dict):
        for feat_name, feat_value in features.items():
            if isinstance(feat_value, Number):
                entry["feature_sums"][feat_name] += feat_value
                entry["feature_counts"][feat_name] += 1

# Build user-level output
user_level_data = []

for author, entry in author_data.items():
    avg_features = {}
    for feat_name in entry["feature_sums"]:
        count = entry["feature_counts"][feat_name]
        if count > 0:
            avg_features[feat_name] = entry["feature_sums"][feat_name] / count

    user_record = {
        "author": author,
        "post_count": entry["post_count"],
        "subreddits": sorted(entry["subreddits"]),
        "features": avg_features,
    }

    # Optionally add the observed time span
    if entry["created_utcs"]:
        user_record["first_created_utc"] = min(entry["created_utcs"])
        user_record["last_created_utc"] = max(entry["created_utcs"])

    # Optionally add the mean original word count
    if entry["raw_word_count_count"] > 0:
        user_record["mean_raw_word_count"] = (
            entry["raw_word_count_sum"] / entry["raw_word_count_count"]
        )

    user_level_data.append(user_record)

# Save aggregated user-level data
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(user_level_data, f, ensure_ascii=False, indent=2)

print(f"Original number of posts: {len(data)}")
print(f"Number of users: {len(user_level_data)}")
print(f"Saved to: {OUTPUT_PATH}")

## Normalising Count-Based Features to Rates per 100 Words

In [ ]:
import json
from pathlib import Path

# Input user-level JSON files to be processed
INPUT_FILES = [
    "depression_user_level.json",
    "normal_user_level.json",
]

# Count-based features to be converted into per-100-word rates
COUNT_FEATURES_TO_NORMALIZE = [
    "definite_articles",
    "indefinite_articles",
    "pronouns",
    "first_person_singular",
    "first_person_plural",
    "second_person",
    "third_person",
    "ambiguous_pronoun_count",
]


def get_denominator_word_count(item: dict) -> float | None:
    """
    Retrieve the denominator word count used for rate normalization.

    Priority:
    1. item["features"]["word_count"]
    2. item["word_count"]

    Returns
    -------
    float or None
        A positive word count if available, otherwise None.
    """
    features = item.get("features", {})
    if isinstance(features, dict):
        wc = features.get("word_count")
        if isinstance(wc, (int, float)) and wc > 0:
            return float(wc)

    wc_outer = item.get("word_count")
    if isinstance(wc_outer, (int, float)) and wc_outer > 0:
        return float(wc_outer)

    return None


def replace_with_per_100_word_features(data: list[dict]) -> list[dict]:
    """
    Replace selected count-based features with per-100-word rate features.

    For each feature listed in COUNT_FEATURES_TO_NORMALIZE:
    - create a new feature named "{feature}_per_100_words"
    - remove the original raw count feature

    Parameters
    ----------
    data : list[dict]
        List of user-level JSON records.

    Returns
    -------
    list[dict]
        Updated records with rate-based features.
    """
    for item in data:
        features = item.get("features", {})
        if not isinstance(features, dict):
            continue

        denom = get_denominator_word_count(item)
        if denom is None or denom <= 0:
            continue

        new_features = {}
        keys_to_delete = []

        for feat in COUNT_FEATURES_TO_NORMALIZE:
            value = features.get(feat)
            if isinstance(value, (int, float)):
                new_features[f"{feat}_per_100_words"] = (float(value) / denom) * 100.0
                keys_to_delete.append(feat)

        features.update(new_features)

        for feat in keys_to_delete:
            del features[feat]

    return data


def main():
    """
    Load each input JSON file, convert selected count features into
    per-100-word rate features, and save the updated output to a new file.
    """
    for input_file in INPUT_FILES:
        input_path = Path(input_file)
        output_path = input_path.with_name(input_path.stem + "_with_rates.json")

        with input_path.open("r", encoding="utf-8") as f:
            data = json.load(f)

        if not isinstance(data, list):
            raise ValueError(f"The top-level JSON structure in {input_file} must be a list.")

        data = replace_with_per_100_word_features(data)

        with output_path.open("w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

        print(f"Saved: {output_path}")


if __name__ == "__main__":
    main()

Saved: depression_user_level_with_rates.json
Saved: normal_user_level_with_rates.json


## A. Feature-Level Statistical Analysis

For each NLP feature, a Shapiro–Wilk test was conducted within each group using the per-100-word normalised values to assess distributional normality. Distributional patterns were also examined visually using spider plots and box plots. Where the feature was not normally distributed, a two-sided Mann–Whitney U test was used to compare the two groups.

In [ ]:
! pip install numpy pandas matplotlib scipy statsmodels

In [ ]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

 
NORMALIZED_COUNT_FEATURE_MAP = {
    "definite_articles": "definite_articles_per_100_words",
    "indefinite_articles": "indefinite_articles_per_100_words",
    "pronouns": "pronouns_per_100_words",
    "first_person_singular": "first_person_singular_per_100_words",
    "first_person_plural": "first_person_plural_per_100_words",
    "second_person": "second_person_per_100_words",
    "third_person": "third_person_per_100_words",
    "ambiguous_pronoun_count": "ambiguous_pronoun_count_per_100_words",
}

COUNT_FEATURES_TO_NORMALIZE = list(NORMALIZED_COUNT_FEATURE_MAP.values())


 
DEPRESSION_JSON = "depression_user_level_with_rates.json"
NORMAL_JSON = "normal_user_level_with_rates.json"

OUTPUT_DIR = Path("feature_visual_analysis")
SPIDER_PNG_1 = OUTPUT_DIR / "group_spider_plot_part1.png"
SPIDER_PNG_2 = OUTPUT_DIR / "group_spider_plot_part2.png"

BOXPLOT_PNG_1 = OUTPUT_DIR / "group_boxplots_part1.png"
BOXPLOT_PNG_2 = OUTPUT_DIR / "group_boxplots_part2.png"
SUMMARY_CSV = OUTPUT_DIR / "feature_group_summary.csv"

RANDOM_SEED = 2025
SHAPIRO_MAX_N = 5000
ALPHA_NORMALITY = 0.05


 
FEATURE_SET_1 = [
    "word_count",
    "sentence_count",
    "mean_sentence_length",
    "semantic_coherence",
    "maximum_similarity",
    "definite_articles_per_100_words",
    "indefinite_articles_per_100_words",
    "pronouns_per_100_words",
    "first_person_singular_per_100_words",
    "first_person_plural_per_100_words",
    "second_person_per_100_words",
    "third_person_per_100_words",
]

FEATURE_SET_2 = [
    "noun_overlap",
    "pronoun_overlap",
    "content_overlap",
    "LCC",
    "LSC",
    "number_of_edges",
    "degree_average",
    "degree_std",
    "LCCr",
    "LSCr",
    "ambiguous_pronoun_count_per_100_words",
]

PLOT_FEATURES = FEATURE_SET_1 + FEATURE_SET_2

FEATURE_LABELS = {
    "word_count": "No. words",
    "sentence_count": "No. sent.s",
    "mean_sentence_length": "Mean sent. length",
    "semantic_coherence": "Coherence",
    "maximum_similarity": "Max. similarity",
    "definite_articles_per_100_words": "Def. articles /100w",
    "indefinite_articles_per_100_words": "Indef. articles /100w",
    "pronouns_per_100_words": "Pronouns /100w",
    "first_person_singular_per_100_words": "1st pers. sing. /100w",
    "first_person_plural_per_100_words": "1st pers. plur. /100w",
    "second_person_per_100_words": "2nd person /100w",
    "third_person_per_100_words": "3rd person /100w",
    "noun_overlap": "Noun overlap",
    "pronoun_overlap": "Pronoun overlap",
    "content_overlap": "Content overlap",
    "LCC": "LCC",
    "LSC": "LSC",
    "number_of_edges": "No. edges",
    "degree_average": "Degree avg.",
    "degree_std": "Degree std.",
    "LCCr": "LCCr",
    "LSCr": "LSCr",
    "ambiguous_pronoun_count_per_100_words": "Ambig. pron. /100w",
}

GROUP_ORDER = ["normal", "depressed"]
GROUP_DISPLAY_NAMES = {
    "normal": "Control",
    "depressed": "Depressed",
}

 
GROUP_COVARIATE = "group"

 
CONTEXT_COVARIATE = "subreddits"

 
def load_group_json(json_path: str, fallback_group_label: str | None = None) -> pd.DataFrame:
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    rows = []
    for item in data:
        row = {}
 
        row["id"] = item.get("id")
        row["text"] = item.get("text")
 
        covariates = item.get("covariates", None)

        if isinstance(covariates, dict):
            # schema A: nested covariates
            for k, v in covariates.items():
                row[k] = v
        else:
            # schema B: top-level covariates
            for k, v in item.items():
                if k not in {"features", "id", "text"}:
                    row[k] = v
 
        if GROUP_COVARIATE not in row and fallback_group_label is not None:
            row[GROUP_COVARIATE] = fallback_group_label

 
        feature_dict = item.get("features", {})
        if isinstance(feature_dict, dict):
            for k, v in feature_dict.items():
                row[k] = v

        rows.append(row)

    return pd.DataFrame(rows)


def build_dataframe(depression_json: str, normal_json: str) -> pd.DataFrame:
    df_dep = load_group_json(depression_json, "depressed")
    df_norm = load_group_json(normal_json, "normal")
    df = pd.concat([df_dep, df_norm], ignore_index=True)
    return df


 
def coerce_numeric(df: pd.DataFrame, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def resolve_plot_features(df: pd.DataFrame, desired_features):
    available = []
    missing = []

    for feat in desired_features:
        if feat in df.columns and pd.api.types.is_numeric_dtype(df[feat]):
            available.append(feat)
        else:
            missing.append(feat)

    if missing:
        print("[Warning] These features are missing or non-numeric and will be skipped:")
        for m in missing:
            print("   -", m)

    return available


def compute_group_means(df: pd.DataFrame, features):
    return df.groupby("group")[features].mean(numeric_only=True)


def compute_feature_ranges(df: pd.DataFrame, features):
    s_vec = []
    e_vec = []

    for feat in features:
        x = pd.to_numeric(df[feat], errors="coerce").dropna().values.astype(float)

        if len(x) == 0:
            s_vec.append(0.0)
            e_vec.append(1.0)
            continue

        s = float(np.min(x))
        e = float(np.max(x))

        if np.isclose(s, e):
            delta = 1.0 if np.isclose(e, 0.0) else abs(e) * 0.1
            s -= delta
            e += delta

        s_vec.append(s)
        e_vec.append(e)

    return np.array(s_vec, dtype=float), np.array(e_vec, dtype=float)


def get_range(start, end, n_levels=5):
    vals = np.linspace(start, end, n_levels + 1)
    return vals[1:]


def shift_svec(s_vec, e_vec):
    return s_vec - (e_vec - s_vec) / 4


def putvec_in_range(result, s_vec, e_vec, n_levels=5):
    result = np.array(result, dtype=float)
    return n_levels * (result - s_vec) / (e_vec - s_vec)


def format_tick_values(values):
    out = []
    for v in values:
        if abs(v) >= 100:
            out.append(f"{v:.0f}")
        elif abs(v) >= 10:
            out.append(f"{v:.1f}")
        else:
            out.append(f"{v:.2f}")
    return out


def compute_summary_stats(x: np.ndarray):
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]

    if len(x) == 0:
        return {
            "n": 0,
            "mean": np.nan,
            "std": np.nan,
            "median": np.nan,
            "q1": np.nan,
            "q3": np.nan,
            "iqr": np.nan,
            "min": np.nan,
            "max": np.nan,
        }

    q1 = np.percentile(x, 25)
    q3 = np.percentile(x, 75)

    return {
        "n": len(x),
        "mean": np.mean(x),
        "std": np.std(x, ddof=1) if len(x) > 1 else np.nan,
        "median": np.median(x),
        "q1": q1,
        "q3": q3,
        "iqr": q3 - q1,
        "min": np.min(x),
        "max": np.max(x),
    }


def safe_shapiro(x: np.ndarray, max_n: int = 5000, random_seed: int = 2025): 
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]

    if len(x) < 3:
        return np.nan, np.nan, len(x), "too_few_samples"

    if np.allclose(x, x[0]):
        return np.nan, np.nan, len(x), "constant_values"

    if len(x) > max_n:
        rng = np.random.default_rng(random_seed)
        idx = rng.choice(len(x), size=max_n, replace=False)
        x_for_test = x[idx]
        note = f"subsampled_to_{max_n}"
    else:
        x_for_test = x
        note = "full_sample"

    W, p = stats.shapiro(x_for_test)
    return W, p, len(x_for_test), note


def shapiro_is_approximately_normal(p_value, note, alpha=0.05): 
    if pd.isna(p_value):
        return False
    if note in {"too_few_samples", "constant_values"}:
        return False
    return p_value > alpha


def safe_welch_ttest(x1: np.ndarray, x2: np.ndarray): 
    x1 = np.asarray(x1, dtype=float)
    x2 = np.asarray(x2, dtype=float)

    x1 = x1[~np.isnan(x1)]
    x2 = x2[~np.isnan(x2)]

    if len(x1) < 2 or len(x2) < 2:
        return np.nan, np.nan, "too_few_samples"

    if np.allclose(x1, x1[0]) and np.allclose(x2, x2[0]):
        return np.nan, np.nan, "constant_values"

    try:
        stat, p = stats.ttest_ind(x1, x2, equal_var=False, nan_policy="omit")
        return stat, p, "welch_t_test"
    except Exception as e:
        return np.nan, np.nan, f"error:{type(e).__name__}"


def safe_mannwhitney(x1: np.ndarray, x2: np.ndarray): 
    x1 = np.asarray(x1, dtype=float)
    x2 = np.asarray(x2, dtype=float)

    x1 = x1[~np.isnan(x1)]
    x2 = x2[~np.isnan(x2)]

    if len(x1) == 0 or len(x2) == 0:
        return np.nan, np.nan, "too_few_samples"

    try:
        stat, p = stats.mannwhitneyu(x1, x2, alternative="two-sided")
        return stat, p, "mannwhitney_u"
    except Exception as e:
        return np.nan, np.nan, f"error:{type(e).__name__}"


def choose_group_comparison_test(
    dep: np.ndarray,
    con: np.ndarray,
    dep_shapiro_p,
    dep_shapiro_note,
    con_shapiro_p,
    con_shapiro_note,
    alpha=0.05,
):
    """
    decision rule:
    - if both groups are approximately Normal -> Welch's t-test
    - otherwise -> two-sided Mann--Whitney U-test
    """
    dep_normal = shapiro_is_approximately_normal(dep_shapiro_p, dep_shapiro_note, alpha)
    con_normal = shapiro_is_approximately_normal(con_shapiro_p, con_shapiro_note, alpha)

    if dep_normal and con_normal:
        stat, p, note = safe_welch_ttest(dep, con)
        return {
            "test_selected": "welch_t_test",
            "test_statistic": stat,
            "test_p_value": p,
            "test_note": note,
            "dep_approx_normal": True,
            "con_approx_normal": True,
        }

    stat, p, note = safe_mannwhitney(dep, con)
    return {
        "test_selected": "mannwhitney_u",
        "test_statistic": stat,
        "test_p_value": p,
        "test_note": note,
        "dep_approx_normal": dep_normal,
        "con_approx_normal": con_normal,
    }


# =========================
# Radar plot
# =========================
class Radar:
    def __init__(self, figure, titles, labels, rect=None):
        if rect is None:
            rect = [0.06, 0.06, 0.88, 0.88]

        self.n = len(titles)
        self.angles = np.arange(0, 360, 360.0 / self.n)

        self.axes = [
            figure.add_axes(rect, projection="polar", label=f"axes{i}")
            for i in range(self.n)
        ]

        self.ax = self.axes[0]

        _, title_texts = self.ax.set_thetagrids(
            self.angles,
            labels=titles,
            fontsize=16
        )

        self.ax.tick_params(axis="x", pad=24)

        for txt in title_texts:
            txt.set_fontsize(16)
            txt.set_bbox(
                dict(
                    boxstyle="square,pad=0.28",
                    facecolor="#f2f2f2",
                    edgecolor="gray",
                    linewidth=1.2
                )
            )

        for ax in self.axes[1:]:
            ax.patch.set_visible(False)
            ax.grid(False)
            ax.xaxis.set_visible(False)

        for ax, angle, label in zip(self.axes, self.angles, labels):
            _, radial_texts = ax.set_rgrids(
                range(1, 6),
                angle=angle,
                labels=label
            )
            ax.spines["polar"].set_visible(False)
            ax.set_ylim(0, 5)

            for rtxt in radial_texts:
                rtxt.set_fontsize(8)
                rtxt.set_color("black")

    def plot(self, values, *args, **kwargs):
        angle = np.deg2rad(np.r_[self.angles, self.angles[0]])
        values = np.r_[values, values[0]]
        self.ax.plot(angle, values, *args, **kwargs)


def plot_group_spider(
    df: pd.DataFrame,
    features,
    output_file: Path,
    title="Average NLP Feature Profiles by Group"
):
    if not features:
        print("[Warning] No valid features for spider plot.")
        return

    group_means = compute_group_means(df, features)
    s_vec, e_vec = compute_feature_ranges(df, features)
    s_vec_shifted = shift_svec(s_vec, e_vec)

    axis_labels = []
    for s, e in zip(s_vec_shifted, e_vec):
        ticks = get_range(s, e, n_levels=5)
        axis_labels.append(format_tick_values(ticks))

    spoke_titles = [FEATURE_LABELS.get(f, f) for f in features]

    fig = plt.figure(figsize=(13.5, 11.5))
    radar = Radar(fig, spoke_titles, axis_labels)

    style_map = {
        "normal": {"color": "#1f77b4", "label": GROUP_DISPLAY_NAMES["normal"]},
        "depressed": {"color": "#d62728", "label": GROUP_DISPLAY_NAMES["depressed"]},
    }

    for group in GROUP_ORDER:
        if group not in group_means.index:
            continue

        values = group_means.loc[group, features].values.astype(float)
        values_scaled = putvec_in_range(values, s_vec_shifted, e_vec, n_levels=5)

        radar.plot(
            values_scaled,
            "-",
            lw=2.8,
            color=style_map[group]["color"],
            alpha=0.9,
            label=style_map[group]["label"],
        )

    radar.ax.legend(
        loc="upper right",
        bbox_to_anchor=(1.18, 1.14),
        frameon=True,
        fontsize=12
    )

    plt.title(title, y=1.12, fontsize=16)
    fig.savefig(output_file, dpi=250, bbox_inches="tight")
    plt.close(fig)


# =========================
# Box plot
# =========================
def plot_group_boxplots(
    df: pd.DataFrame,
    features,
    output_file: Path,
    ncols=4,
    title="Box-plots of NLP Measures by Group"
):
    if not features:
        print("[Warning] No valid features for box plots.")
        return

    n_features = len(features)
    ncols = min(ncols, n_features)
    nrows = math.ceil(n_features / ncols)

    fig, axes = plt.subplots(nrows, ncols, figsize=(4.3 * ncols, 3.5 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for i, feat in enumerate(features):
        ax = axes[i]

        data = []
        labels = []
        for g in GROUP_ORDER:
            x = pd.to_numeric(
                df.loc[df["group"] == g, feat],
                errors="coerce"
            ).dropna().values
            data.append(x)
            labels.append(GROUP_DISPLAY_NAMES.get(g, g))

        bp = ax.boxplot(
            data,
            labels=labels,
            patch_artist=True,
            widths=0.6,
            showfliers=False,
            medianprops=dict(color="red", linewidth=1.5),
            whiskerprops=dict(color="black"),
            capprops=dict(color="black"),
            boxprops=dict(edgecolor="black", linewidth=1.0),
        )

        for patch in bp["boxes"]:
            patch.set_facecolor("#cfcfcf")

        ax.set_title(FEATURE_LABELS.get(feat, feat), fontsize=10)
        ax.grid(axis="y", alpha=0.25)

    for j in range(n_features, len(axes)):
        fig.delaxes(axes[j])

    plt.suptitle(title, fontsize=14, y=0.99)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig(output_file, dpi=250)
    plt.close(fig)


# =========================
# Summary table
# =========================
def build_summary_table(df: pd.DataFrame, features):
    rows = []

    for feat in features:
        dep = pd.to_numeric(
            df.loc[df["group"] == "depressed", feat], errors="coerce"
        ).dropna().values.astype(float)

        con = pd.to_numeric(
            df.loc[df["group"] == "normal", feat], errors="coerce"
        ).dropna().values.astype(float)

        dep_stats = compute_summary_stats(dep)
        con_stats = compute_summary_stats(con)

        dep_W, dep_p, dep_n_used, dep_note = safe_shapiro(
            dep,
            max_n=SHAPIRO_MAX_N,
            random_seed=RANDOM_SEED,
        )
        con_W, con_p, con_n_used, con_note = safe_shapiro(
            con,
            max_n=SHAPIRO_MAX_N,
            random_seed=RANDOM_SEED,
        )

        comparison = choose_group_comparison_test(
            dep=dep,
            con=con,
            dep_shapiro_p=dep_p,
            dep_shapiro_note=dep_note,
            con_shapiro_p=con_p,
            con_shapiro_note=con_note,
            alpha=ALPHA_NORMALITY,
        )

        rows.append({
            "feature": feat,
            "label": FEATURE_LABELS.get(feat, feat),

            "n_depressed": dep_stats["n"],
            "mean_depressed": dep_stats["mean"],
            "std_depressed": dep_stats["std"],
            "median_depressed": dep_stats["median"],
            "q1_depressed": dep_stats["q1"],
            "q3_depressed": dep_stats["q3"],
            "iqr_depressed": dep_stats["iqr"],
            "min_depressed": dep_stats["min"],
            "max_depressed": dep_stats["max"],

            "shapiro_W_depressed": dep_W,
            "shapiro_p_depressed": dep_p,
            "shapiro_n_used_depressed": dep_n_used,
            "shapiro_note_depressed": dep_note,
            "approx_normal_depressed": comparison["dep_approx_normal"],

            "n_normal": con_stats["n"],
            "mean_normal": con_stats["mean"],
            "std_normal": con_stats["std"],
            "median_normal": con_stats["median"],
            "q1_normal": con_stats["q1"],
            "q3_normal": con_stats["q3"],
            "iqr_normal": con_stats["iqr"],
            "min_normal": con_stats["min"],
            "max_normal": con_stats["max"],

            "shapiro_W_normal": con_W,
            "shapiro_p_normal": con_p,
            "shapiro_n_used_normal": con_n_used,
            "shapiro_note_normal": con_note,
            "approx_normal_normal": comparison["con_approx_normal"],

            "group_test_selected": comparison["test_selected"],
            "group_test_statistic": comparison["test_statistic"],
            "group_test_p_value": comparison["test_p_value"],
            "group_test_note": comparison["test_note"],
        })

    return pd.DataFrame(rows)


 
def main():
    OUTPUT_DIR.mkdir(exist_ok=True)

    df = build_dataframe(DEPRESSION_JSON, NORMAL_JSON)
    df = coerce_numeric(df, PLOT_FEATURES)

    feature_set_1 = resolve_plot_features(df, FEATURE_SET_1)
    feature_set_2 = resolve_plot_features(df, FEATURE_SET_2)

    plot_features = feature_set_1 + feature_set_2

    if not plot_features:
        print("No valid numeric features found. Please check your feature names.")
        return

    print("[Info] Feature set 1:")
    for f in feature_set_1:
        print("  -", f)

    print("[Info] Feature set 2:")
    for f in feature_set_2:
        print("  -", f)

    # 1) spider plots
    if feature_set_1:
        plot_group_spider(
            df,
            feature_set_1,
            SPIDER_PNG_1,
            title="Average NLP Feature Profiles by Group (Part 1)"
        )

    if feature_set_2:
        plot_group_spider(
            df,
            feature_set_2,
            SPIDER_PNG_2,
            title="Average NLP Feature Profiles by Group (Part 2)"
        )

    # 2) box plots
    if feature_set_1:
        plot_group_boxplots(df, feature_set_1, BOXPLOT_PNG_1, ncols=4)

    if feature_set_2:
        plot_group_boxplots(df, feature_set_2, BOXPLOT_PNG_2, ncols=4)

    # 3) summary + Shapiro--Wilk + feature-wise adaptive group comparison
    summary_df = build_summary_table(df, plot_features)
    summary_df.to_csv(SUMMARY_CSV, index=False, encoding="utf-8-sig")

    print("Done.")
    if feature_set_1:
        print(f"Spider plot 1 saved to: {SPIDER_PNG_1.resolve()}")
        print(f"Box plots 1 saved to: {BOXPLOT_PNG_1.resolve()}")
    if feature_set_2:
        print(f"Spider plot 2 saved to: {SPIDER_PNG_2.resolve()}")
        print(f"Box plots 2 saved to: {BOXPLOT_PNG_2.resolve()}")

    print(f"Summary table saved to: {SUMMARY_CSV.resolve()}")

    if not summary_df.empty:
        print("\n[Info] Selected group comparison tests:")
        print(summary_df["group_test_selected"].value_counts(dropna=False))


if __name__ == "__main__":
    main()

[Info] Feature set 1:
  - word_count
  - sentence_count
  - mean_sentence_length
  - semantic_coherence
  - maximum_similarity
  - definite_articles_per_100_words
  - indefinite_articles_per_100_words
  - pronouns_per_100_words
  - first_person_singular_per_100_words
  - first_person_plural_per_100_words
  - second_person_per_100_words
  - third_person_per_100_words
[Info] Feature set 2:
  - noun_overlap
  - pronoun_overlap
  - content_overlap
  - LCC
  - LSC
  - number_of_edges
  - degree_average
  - degree_std
  - LCCr
  - LSCr
  - ambiguous_pronoun_count_per_100_words


C:\Users\guoji\AppData\Local\Temp\ipykernel_39104\2636325268.py:572: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(
C:\Users\guoji\AppData\Local\Temp\ipykernel_39104\2636325268.py:572: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(
C:\Users\guoji\AppData\Local\Temp\ipykernel_39104\2636325268.py:572: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(
C:\Users\guoji\AppData\Local\Temp\ipykernel_39104\2636325268.py:572: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped 

Done.
Spider plot 1 saved to: C:\Users\guoji\Downloads\毕设code\feature_visual_analysis\group_spider_plot_part1.png
Box plots 1 saved to: C:\Users\guoji\Downloads\毕设code\feature_visual_analysis\group_boxplots_part1.png
Spider plot 2 saved to: C:\Users\guoji\Downloads\毕设code\feature_visual_analysis\group_spider_plot_part2.png
Box plots 2 saved to: C:\Users\guoji\Downloads\毕设code\feature_visual_analysis\group_boxplots_part2.png
Summary table saved to: C:\Users\guoji\Downloads\毕设code\feature_visual_analysis\feature_group_summary.csv

[Info] Selected group comparison tests:
group_test_selected
mannwhitney_u    23
Name: count, dtype: int64


## Linear Regression Analysis of Feature Associations

A linear regression model of the form `feature_y ~ feature_x + group` was used to assess whether the association between two features remained after adjusting for group membership.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf


 
DEPRESSION_JSON = "depression_user_level_with_rates.json"
NORMAL_JSON = "normal_user_level_with_rates.json"

OUTPUT_DIR = Path("feature_relationship_results")
OUTPUT_DIR.mkdir(exist_ok=True)

 
GROUP_COVARIATE = "group"

 
CONTEXT_COVARIATE = "subreddits"
 
 
FEATURES = [
    "word_count",
    "sentence_count",
    "mean_sentence_length",
    "semantic_coherence",
    "maximum_similarity",

    "definite_articles_per_100_words",
    "indefinite_articles_per_100_words",
    "pronouns_per_100_words",
    "first_person_singular_per_100_words",
    "first_person_plural_per_100_words",
    "second_person_per_100_words",
    "third_person_per_100_words",

    "noun_overlap",
    "pronoun_overlap",
    "content_overlap",

    "LCC",
    "LSC",
    "number_of_edges",
    "degree_average",
    "degree_std",
    "LCCr",
    "LSCr",

    "ambiguous_pronoun_count_per_100_words",
]

FEATURE_LABELS = {
    "word_count": "No. words",
    "sentence_count": "No. sent.s",
    "mean_sentence_length": "Mean sent. length",
    "semantic_coherence": "Coherence",
    "maximum_similarity": "Max. similarity",

    "definite_articles_per_100_words": "Def. articles /100w",
    "indefinite_articles_per_100_words": "Indef. articles /100w",
    "pronouns_per_100_words": "Pronouns /100w",
    "first_person_singular_per_100_words": "1st pers. sing. /100w",
    "first_person_plural_per_100_words": "1st pers. plur. /100w",
    "second_person_per_100_words": "2nd person /100w",
    "third_person_per_100_words": "3rd person /100w",

    "noun_overlap": "Noun overlap",
    "pronoun_overlap": "Pronoun overlap",
    "content_overlap": "Content overlap",

    "LCC": "LCC",
    "LSC": "LSC",
    "number_of_edges": "No. edges",
    "degree_average": "Degree avg.",
    "degree_std": "Degree std.",
    "LCCr": "LCCr",
    "LSCr": "LSCr",

    "ambiguous_pronoun_count_per_100_words": "Ambig. pron. /100w",
}

P_THRESHOLD = 0.01
MIN_ROWS = 20


 
def load_group_json(json_path: str, fallback_group_label: str | None = None) -> pd.DataFrame:
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    rows = []
    for item in data:
        row = {} 

        row["id"] = item.get("id")
        row["text"] = item.get("text")
 
        covariates = item.get("covariates", None)

        if isinstance(covariates, dict):
            # schema A: nested covariates
            for k, v in covariates.items():
                row[k] = v
        else:
            # schema B: top-level covariates
            for k, v in item.items():
                if k not in {"features", "id", "text"}:
                    row[k] = v
 
        if GROUP_COVARIATE not in row and fallback_group_label is not None:
            row[GROUP_COVARIATE] = fallback_group_label

 
        feature_dict = item.get("features", {})
        if isinstance(feature_dict, dict):
            for k, v in feature_dict.items():
                row[k] = v

        rows.append(row)

    return pd.DataFrame(rows)


def build_dataframe(dep_path: str, norm_path: str) -> pd.DataFrame:
    df_dep = load_group_json(dep_path, "depressed")
    df_norm = load_group_json(norm_path, "normal")
    df = pd.concat([df_dep, df_norm], ignore_index=True)

    df["group"] = df["group"].astype(str)
    df["group_binary"] = (df["group"] == "depressed").astype(int)

    return df


def coerce_numeric(df: pd.DataFrame, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def resolve_features(df: pd.DataFrame, features):
    ok = []
    missing = []
    for f in features:
        if f in df.columns and pd.api.types.is_numeric_dtype(df[f]):
            ok.append(f)
        else:
            missing.append(f)

    if missing:
        print("[Warning] Missing/non-numeric features skipped:")
        for m in missing:
            print("  -", m)

    return ok





# =========================
# pairwise linear regression
# feature_y ~ feature_x + group
# =========================
def pairwise_regression_matrix(df: pd.DataFrame, features: list[str]):
    n = len(features)

    t_mat = pd.DataFrame(np.nan, index=features, columns=features)
    p_mat = pd.DataFrame(np.nan, index=features, columns=features)
    coef_mat = pd.DataFrame(np.nan, index=features, columns=features)

    long_rows = []

    for i, fx in enumerate(features):
        for j, fy in enumerate(features):
            if i == j:
                t_mat.loc[fy, fx] = np.nan
                p_mat.loc[fy, fx] = np.nan
                coef_mat.loc[fy, fx] = np.nan
                continue

            sub = df[[fx, fy, "group_binary"]].copy()
            sub[fx] = pd.to_numeric(sub[fx], errors="coerce")
            sub[fy] = pd.to_numeric(sub[fy], errors="coerce")
            sub = sub.dropna()

            if len(sub) < MIN_ROWS:
                continue
            if sub[fx].nunique() < 2 or sub[fy].nunique() < 2:
                continue
            if sub["group_binary"].nunique() < 2:
                continue

            formula = f'Q("{fy}") ~ Q("{fx}") + group_binary'

            try:
                model = smf.ols(formula, data=sub).fit()
                term = f'Q("{fx}")'

                coef = float(model.params.get(term, np.nan))
                tval = float(model.tvalues.get(term, np.nan))
                pval = float(model.pvalues.get(term, np.nan))

                coef_mat.loc[fy, fx] = coef
                t_mat.loc[fy, fx] = tval
                p_mat.loc[fy, fx] = pval

                long_rows.append({
                    "feature_x": fx,
                    "feature_y": fy,
                    "coef_x": coef,
                    "t_x": tval,
                    "p_x": pval,
                    "n": int(model.nobs),
                    "r_squared": float(model.rsquared),
                })

            except Exception as e:
                long_rows.append({
                    "feature_x": fx,
                    "feature_y": fy,
                    "coef_x": np.nan,
                    "t_x": np.nan,
                    "p_x": np.nan,
                    "n": len(sub),
                    "r_squared": np.nan,
                    "error": str(e),
                })

    long_df = pd.DataFrame(long_rows)
    return coef_mat, t_mat, p_mat, long_df


# =========================
# draw heat map
# =========================
def plot_heatmap(matrix: pd.DataFrame, title: str, output_path: Path, cmap="coolwarm", vlim=(-10, 10)):
    labels = [FEATURE_LABELS.get(c, c) for c in matrix.columns]

    fig, ax = plt.subplots(figsize=(12, 10))
    data = matrix.values.astype(float)

    vmin, vmax = vlim
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)

    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=60, ha="right", fontsize=9)
    ax.set_yticklabels(labels, fontsize=9)

    ax.set_title(title, fontsize=14)
    cbar = plt.colorbar(im, ax=ax)
    cbar.ax.set_ylabel("T-statistic", rotation=90)

    plt.tight_layout()
    plt.savefig(output_path, dpi=250, bbox_inches="tight")
    plt.close(fig)


def plot_significant_heatmap(t_mat: pd.DataFrame, p_mat: pd.DataFrame, p_thresh: float, output_path: Path, vlim=(-10, 10)):
    sig_t = t_mat.copy()
    sig_t[p_mat >= p_thresh] = np.nan

    labels = [FEATURE_LABELS.get(c, c) for c in sig_t.columns]

    fig, ax = plt.subplots(figsize=(12, 10))
    data = sig_t.values.astype(float)

    vmin, vmax = vlim

    cmap = plt.cm.coolwarm.copy()
    cmap.set_bad(color="white")

    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)

    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=60, ha="right", fontsize=9)
    ax.set_yticklabels(labels, fontsize=9)

    ax.set_title(f"T-statistic heat map (only P < {p_thresh})", fontsize=14)
    cbar = plt.colorbar(im, ax=ax)
    cbar.ax.set_ylabel("T-statistic", rotation=90)

    plt.tight_layout()
    plt.savefig(output_path, dpi=250, bbox_inches="tight")
    plt.close(fig)


 
def main():
    df = build_dataframe(DEPRESSION_JSON, NORMAL_JSON)
    df = coerce_numeric(df, FEATURES)

    features = resolve_features(df, FEATURES)
    if not features:
        print("No usable features found.")
        return


    df_model = df.copy()
    print(f"[Info] Using data: {len(df_model)} rows")

    coef_mat, t_mat, p_mat, long_df = pairwise_regression_matrix(df_model, features)


    coef_mat.to_csv(OUTPUT_DIR / "pairwise_coef_matrix.csv", encoding="utf-8-sig")
    t_mat.to_csv(OUTPUT_DIR / "pairwise_t_matrix.csv", encoding="utf-8-sig")
    p_mat.to_csv(OUTPUT_DIR / "pairwise_p_matrix.csv", encoding="utf-8-sig")
    long_df.to_csv(OUTPUT_DIR / "pairwise_regression_long.csv", index=False, encoding="utf-8-sig")

    
    plot_heatmap(
        t_mat,
        title="Relationships between NLP measures (T-statistics)",
        output_path=OUTPUT_DIR / "heatmap_t_statistics.png",
    )

    plot_significant_heatmap(
        t_mat,
        p_mat,
        p_thresh=P_THRESHOLD,
        output_path=OUTPUT_DIR / f"heatmap_t_statistics_p_lt_{str(P_THRESHOLD).replace('.', '_')}.png",
    )

    print("Done.")
    print(f"Saved to: {OUTPUT_DIR.resolve()}")


if __name__ == "__main__":
    main()

[Info] Using data: 2167 rows
Done.
Saved to: C:\Users\guoji\Downloads\毕设code\feature_relationship_results


## Covariate-Adjusted Sensitivity Analysis

To assess whether group membership remained associated with each feature after adjustment for covariates, a covariate-adjusted sensitivity analysis was conducted. In the current dataset, the only covariate included in this stage was `subreddit`.

For features considered suitable for a positively skewed response distribution, a Gamma GLM was fitted. For features not considered suitable for a Gamma model, an OLS model with HC3 robust standard errors was used instead. This analysis was intended to examine whether the observed group effect remained after controlling for subreddit.

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests


 
DEPRESSION_JSON = "depression_user_level_with_rates.json"
NORMAL_JSON = "normal_user_level_with_rates.json"

OUTPUT_DIR = Path("covariate_sensitivity_results")
OUTPUT_DIR.mkdir(exist_ok=True)

 
STAGE1_SUMMARY_CSV = Path("feature_visual_analysis") / "feature_group_summary.csv"

MIN_ROWS_PER_MODEL = 20
MIN_ROWS_PER_GROUP = 5

 
GROUP_COVARIATE = "group"

 
CONTEXT_COVARIATE = "primary_subreddit"

 
PRIMARY_CONTEXT_SOURCE = "subreddits"

 
ALL_COVARIATES_MODE_KEYS = {"covariate", "covariates"}

 
TOP_N_CONTEXT_LEVELS = 20

 
SKEWNESS_MIN_FOR_GAMMA = 0.5
NEAR_ZERO_PROP_MAX = 0.25
NEAR_ZERO_RELATIVE_TO_MEDIAN = 0.05
UNIT_INTERVAL_EPS = 1e-9

 
POSITIVE_GROUP_LABEL = "depressed"
NEGATIVE_GROUP_LABEL = "normal"


# =========================
# Features
# =========================
FEATURES = [
    "word_count",
    "sentence_count",
    "mean_sentence_length",
    "semantic_coherence",
    "maximum_similarity",

    "definite_articles_per_100_words",
    "indefinite_articles_per_100_words",
    "pronouns_per_100_words",
    "first_person_singular_per_100_words",
    "first_person_plural_per_100_words",
    "second_person_per_100_words",
    "third_person_per_100_words",

    "noun_overlap",
    "pronoun_overlap",
    "content_overlap",

    "LCC",
    "LSC",
    "number_of_edges",
    "degree_average",
    "degree_std",
    "LCCr",
    "LSCr",

    "ambiguous_pronoun_count_per_100_words",
]

 
GAMMA_CANDIDATE_FEATURES = {
    "word_count",
    "sentence_count",
    "mean_sentence_length",

    "definite_articles_per_100_words",
    "indefinite_articles_per_100_words",
    "pronouns_per_100_words",
    "first_person_singular_per_100_words",
    "first_person_plural_per_100_words",
    "second_person_per_100_words",
    "third_person_per_100_words",
    "ambiguous_pronoun_count_per_100_words",

    "LCC",
    "LSC",
    "number_of_edges",
    "degree_average",
    "degree_std",
}

 
FORCE_OLS_FEATURES = {
    "semantic_coherence",
    "maximum_similarity",
    "noun_overlap",
    "pronoun_overlap",
    "content_overlap",
    "LCCr",
    "LSCr",
}


 
def _as_bool(x) -> bool:
    if pd.isna(x):
        return False
    if isinstance(x, bool):
        return x
    if isinstance(x, (int, float)):
        return bool(x)
    s = str(x).strip().lower()
    return s in {"true", "1", "yes"}


def load_stage1_normality_map(csv_path: Path) -> dict:
    if not csv_path.exists():
        print(f"[Warning] Stage 1 summary CSV not found: {csv_path}")
        return {}

    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"[Warning] Failed to read Stage 1 summary CSV: {e}")
        return {}

    required_cols = {
        "feature",
        "approx_normal_depressed",
        "approx_normal_normal",
    }
    if not required_cols.issubset(df.columns):
        print("[Warning] Stage 1 summary CSV missing required columns; normality map disabled.")
        return {}

    out = {}
    for _, row in df.iterrows():
        feat = row["feature"]
        dep_norm = _as_bool(row["approx_normal_depressed"])
        con_norm = _as_bool(row["approx_normal_normal"])
        out[str(feat)] = {
            "approx_normal_depressed": dep_norm,
            "approx_normal_normal": con_norm,
            "both_approx_normal": dep_norm and con_norm,
        }
    return out


def load_group_json(json_path: str, fallback_group_label: str | None = None) -> pd.DataFrame:
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    rows = []
    for item in data:
        row = {}
 
        row["id"] = item.get("id")
        row["text"] = item.get("text")
 
        covariates = item.get("covariates", None)
        if isinstance(covariates, dict):
            for k, v in covariates.items():
                row[k] = v
 
        for k, v in item.items():
            if k not in {"features", "id", "text", "covariates"} and k not in row:
                row[k] = v
 
        if GROUP_COVARIATE not in row and fallback_group_label is not None:
            row[GROUP_COVARIATE] = fallback_group_label
 
        feature_dict = item.get("features", {})
        if isinstance(feature_dict, dict):
            for k, v in feature_dict.items():
                row[k] = v

        rows.append(row)

    return pd.DataFrame(rows)


def derive_primary_context_value(x): 

    if isinstance(x, dict):
        numeric_items = []
        for k, v in x.items():
            try:
                numeric_items.append((str(k), float(v)))
            except Exception:
                continue
        if numeric_items:
            numeric_items.sort(key=lambda t: t[1], reverse=True)
            return numeric_items[0][0]
        return "UNKNOWN"

    if isinstance(x, (list, tuple, set)):
        values = [str(v) for v in x if str(v).strip()]
        if len(values) == 0:
            return "UNKNOWN"
        return values[0]

    if pd.isna(x):
        return "UNKNOWN"

    s = str(x).strip()
    if not s:
        return "UNKNOWN"
    return s


def build_dataframe(dep_path: str, norm_path: str) -> pd.DataFrame:
    df_dep = load_group_json(dep_path, fallback_group_label=POSITIVE_GROUP_LABEL)
    df_norm = load_group_json(norm_path, fallback_group_label=NEGATIVE_GROUP_LABEL)
    df = pd.concat([df_dep, df_norm], ignore_index=True)

    if GROUP_COVARIATE not in df.columns:
        raise ValueError(f"Required group covariate '{GROUP_COVARIATE}' not found in input data.")

    df[GROUP_COVARIATE] = df[GROUP_COVARIATE].astype(str)
    df["group_binary"] = (df[GROUP_COVARIATE] == POSITIVE_GROUP_LABEL).astype(int)
 
    if CONTEXT_COVARIATE not in df.columns:
        if PRIMARY_CONTEXT_SOURCE in df.columns:
            df[CONTEXT_COVARIATE] = df[PRIMARY_CONTEXT_SOURCE].apply(derive_primary_context_value)
            print(
                f"[Info] '{CONTEXT_COVARIATE}' not found; derived from '{PRIMARY_CONTEXT_SOURCE}'. "
                f"If '{PRIMARY_CONTEXT_SOURCE}' is a list, the first element is treated as the primary context."
            )
        else:
            raise ValueError(
                f"Neither '{CONTEXT_COVARIATE}' nor fallback source '{PRIMARY_CONTEXT_SOURCE}' "
                f"was found in input data."
            )

    return df


def coerce_numeric(df: pd.DataFrame, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def resolve_features(df: pd.DataFrame, desired_features):
    ok = []
    missing = []
    for f in desired_features:
        if f in df.columns and pd.api.types.is_numeric_dtype(df[f]):
            ok.append(f)
        else:
            missing.append(f)

    if missing:
        print("[Warning] Missing/non-numeric features skipped:")
        for m in missing:
            print("  -", m)

    return ok


def normalise_context_value(x):
    if isinstance(x, list):
        if len(x) == 0:
            return "UNKNOWN"
        return "|".join(map(str, x))
    if isinstance(x, dict):
        if len(x) == 0:
            return "UNKNOWN"
        return json.dumps(x, ensure_ascii=False, sort_keys=True)
    if pd.isna(x):
        return "UNKNOWN"
    return str(x)


def column_is_numeric_like(series: pd.Series) -> bool:
    non_null = series.dropna()
    if len(non_null) == 0:
        return False
    converted = pd.to_numeric(non_null, errors="coerce")
    return converted.notna().all()


def get_requested_covariate_names(df: pd.DataFrame) -> list[str]:
    if CONTEXT_COVARIATE not in df.columns:
        raise ValueError(
            f"Required context covariate '{CONTEXT_COVARIATE}' not found in input data."
        )
    return [CONTEXT_COVARIATE]


def prepare_model_covariates(
    df: pd.DataFrame,
    covariate_names: list[str],
    top_n_levels: int = 20,
): 
    
    df = df.copy()
    covariate_cols = []
    covariate_terms = []

    for col in covariate_names:
        if col not in df.columns:
            continue
        if col == GROUP_COVARIATE:
            continue

        series = df[col]

        # numeric covariate
        if pd.api.types.is_numeric_dtype(series) or column_is_numeric_like(series):
            df[col] = pd.to_numeric(df[col], errors="coerce")
            covariate_cols.append(col)
            covariate_terms.append(f'Q("{col}")')
            continue

        # categorical covariate
        major_col = f"{col}__major"
        df[major_col] = df[col].apply(normalise_context_value)

        top_levels = (
            df[major_col]
            .value_counts(dropna=False)
            .head(top_n_levels)
            .index
            .tolist()
        )

        df[major_col] = np.where(
            df[major_col].isin(top_levels),
            df[major_col],
            "OTHER"
        )

        covariate_cols.append(major_col)
        covariate_terms.append(f'C(Q("{major_col}"))')

    return df, covariate_cols, covariate_terms


def build_model_formula(feature: str, covariate_terms: list[str]) -> str:
    rhs = ["group_binary"] + covariate_terms
    rhs_str = " + ".join(rhs)
    return f'Q("{feature}") ~ {rhs_str}'


def validate_feature_data(
    df: pd.DataFrame,
    feature: str,
    covariate_cols: list[str],
) -> pd.DataFrame | None:
    cols = [GROUP_COVARIATE, "group_binary", feature] + covariate_cols
    cols = [c for c in cols if c in df.columns]

    sub = df[cols].copy()
    sub[feature] = pd.to_numeric(sub[feature], errors="coerce")
    sub = sub.dropna()

    if len(sub) < MIN_ROWS_PER_MODEL:
        return None

    group_counts = sub[GROUP_COVARIATE].value_counts()
    if POSITIVE_GROUP_LABEL not in group_counts or NEGATIVE_GROUP_LABEL not in group_counts:
        return None
    if group_counts[POSITIVE_GROUP_LABEL] < MIN_ROWS_PER_GROUP or group_counts[NEGATIVE_GROUP_LABEL] < MIN_ROWS_PER_GROUP:
        return None

    if sub[feature].nunique() < 2:
        return None

    return sub


def gamma_suitability_report(
    sub: pd.DataFrame,
    feature: str,
    stage1_normality_map: dict | None = None,
) -> dict:
    y = pd.to_numeric(sub[feature], errors="coerce").dropna().values.astype(float)

    report = {
        "feature": feature,
        "gamma_candidate_feature": feature in GAMMA_CANDIDATE_FEATURES,
        "stage1_both_approx_normal": False,
        "strictly_positive": False,
        "non_constant": False,
        "bounded_0_1_like": False,
        "skewness": np.nan,
        "right_skewed_enough": False,
        "near_zero_threshold": np.nan,
        "near_zero_prop": np.nan,
        "near_zero_ok": False,
        "gamma_ok": False,
        "gamma_reason": "",
    }

    if len(y) < MIN_ROWS_PER_MODEL:
        report["gamma_reason"] = "too_few_rows"
        return report

    if feature not in GAMMA_CANDIDATE_FEATURES:
        report["gamma_reason"] = "not_in_gamma_candidate_set"
        return report

    if stage1_normality_map is not None and feature in stage1_normality_map:
        report["stage1_both_approx_normal"] = bool(
            stage1_normality_map[feature].get("both_approx_normal", False)
        )
        if report["stage1_both_approx_normal"]:
            report["gamma_reason"] = "approximately_normal_in_stage1"
            return report

    if np.any(y <= 0):
        report["gamma_reason"] = "non_positive_values_present"
        return report
    report["strictly_positive"] = True

    if np.nanstd(y) == 0:
        report["gamma_reason"] = "constant_values"
        return report
    report["non_constant"] = True

    y_min = float(np.nanmin(y))
    y_max = float(np.nanmax(y))
    if y_min >= -UNIT_INTERVAL_EPS and y_max <= 1 + UNIT_INTERVAL_EPS:
        report["bounded_0_1_like"] = True
        report["gamma_reason"] = "bounded_unit_interval_like"
        return report

    skewness = float(pd.Series(y).skew())
    report["skewness"] = skewness
    if not np.isfinite(skewness) or skewness < SKEWNESS_MIN_FOR_GAMMA:
        report["gamma_reason"] = "not_right_skewed_enough"
        return report
    report["right_skewed_enough"] = True

    median_y = float(np.nanmedian(y))
    near_zero_threshold = max(1e-8, median_y * NEAR_ZERO_RELATIVE_TO_MEDIAN)
    near_zero_prop = float(np.mean(y <= near_zero_threshold))

    report["near_zero_threshold"] = near_zero_threshold
    report["near_zero_prop"] = near_zero_prop

    if near_zero_prop > NEAR_ZERO_PROP_MAX:
        report["gamma_reason"] = "too_much_mass_near_zero"
        return report
    report["near_zero_ok"] = True

    report["gamma_ok"] = True
    report["gamma_reason"] = "ok"
    return report

 
def fit_ols_hc3(
    df: pd.DataFrame,
    feature: str,
    covariate_cols: list[str],
    covariate_terms: list[str],
    stage1_normality_map: dict | None = None,
) -> dict | None:
    sub = validate_feature_data(df, feature, covariate_cols=covariate_cols)
    if sub is None:
        return None

    formula = build_model_formula(feature, covariate_terms)

    try:
        model = smf.ols(formula, data=sub).fit(cov_type="HC3")
    except Exception as e:
        return {
            "feature": feature,
            "model_type": "ols_hc3",
            "status": f"fit_failed: {e}",
        }

    term = "group_binary"

    try:
        ci = model.conf_int().loc[term]
        ci_low, ci_high = float(ci[0]), float(ci[1])
    except Exception:
        ci_low, ci_high = np.nan, np.nan

    stage1_both_approx_normal = False
    if stage1_normality_map is not None and feature in stage1_normality_map:
        stage1_both_approx_normal = bool(
            stage1_normality_map[feature].get("both_approx_normal", False)
        )

    return {
        "feature": feature,
        "model_type": "ols_hc3",
        "status": "ok",
        "n": int(model.nobs),
        "coef_group": float(model.params.get(term, np.nan)),
        "se_group": float(model.bse.get(term, np.nan)),
        "z_or_t_group": float(model.tvalues.get(term, np.nan)),
        "p_group": float(model.pvalues.get(term, np.nan)),
        "ci_low": ci_low,
        "ci_high": ci_high,
        "effect_interpretation": "additive",
        "lambda_or_irr": np.nan,
        "lambda_or_irr_ci_low": np.nan,
        "lambda_or_irr_ci_high": np.nan,
        "aic": float(getattr(model, "aic", np.nan)),
        "bic": float(getattr(model, "bic", np.nan)),
        "stage1_both_approx_normal": stage1_both_approx_normal,
        "gamma_reason": np.nan,
        "gamma_skewness": np.nan,
        "gamma_near_zero_prop": np.nan,
        "gamma_bounded_0_1_like": np.nan,
        "selection_reason": np.nan,
    }


def fit_gamma_glm(
    df: pd.DataFrame,
    feature: str,
    covariate_cols: list[str],
    covariate_terms: list[str],
    stage1_normality_map: dict | None = None,
) -> dict | None:
    sub = validate_feature_data(df, feature, covariate_cols=covariate_cols)
    if sub is None:
        return None

    gamma_check = gamma_suitability_report(
        sub=sub,
        feature=feature,
        stage1_normality_map=stage1_normality_map,
    )

    if not gamma_check["gamma_ok"]:
        return {
            "feature": feature,
            "model_type": "gamma_glm_log",
            "status": "incompatible_with_gamma",
            "gamma_reason": gamma_check["gamma_reason"],
            "stage1_both_approx_normal": gamma_check["stage1_both_approx_normal"],
            "gamma_skewness": gamma_check["skewness"],
            "gamma_near_zero_prop": gamma_check["near_zero_prop"],
            "gamma_bounded_0_1_like": gamma_check["bounded_0_1_like"],
            "selection_reason": np.nan,
        }

    formula = build_model_formula(feature, covariate_terms)

    try:
        model = smf.glm(
            formula=formula,
            data=sub,
            family=sm.families.Gamma(link=sm.families.links.log())
        ).fit()
    except Exception as e:
        return {
            "feature": feature,
            "model_type": "gamma_glm_log",
            "status": f"fit_failed: {e}",
            "gamma_reason": "fit_failed",
            "stage1_both_approx_normal": gamma_check["stage1_both_approx_normal"],
            "gamma_skewness": gamma_check["skewness"],
            "gamma_near_zero_prop": gamma_check["near_zero_prop"],
            "gamma_bounded_0_1_like": gamma_check["bounded_0_1_like"],
            "selection_reason": np.nan,
        }

    term = "group_binary"

    try:
        ci = model.conf_int().loc[term]
        ci_low, ci_high = float(ci[0]), float(ci[1])
    except Exception:
        ci_low, ci_high = np.nan, np.nan

    coef = float(model.params.get(term, np.nan))

    return {
        "feature": feature,
        "model_type": "gamma_glm_log",
        "status": "ok",
        "n": int(model.nobs),
        "coef_group": coef,
        "se_group": float(model.bse.get(term, np.nan)),
        "z_or_t_group": float(model.tvalues.get(term, np.nan)),
        "p_group": float(model.pvalues.get(term, np.nan)),
        "ci_low": ci_low,
        "ci_high": ci_high,
        "effect_interpretation": "multiplicative",
        "lambda_or_irr": float(np.exp(coef)) if not pd.isna(coef) else np.nan,
        "lambda_or_irr_ci_low": float(np.exp(ci_low)) if not pd.isna(ci_low) else np.nan,
        "lambda_or_irr_ci_high": float(np.exp(ci_high)) if not pd.isna(ci_high) else np.nan,
        "aic": float(getattr(model, "aic", np.nan)),
        "bic": np.nan,
        "stage1_both_approx_normal": gamma_check["stage1_both_approx_normal"],
        "gamma_reason": gamma_check["gamma_reason"],
        "gamma_skewness": gamma_check["skewness"],
        "gamma_near_zero_prop": gamma_check["near_zero_prop"],
        "gamma_bounded_0_1_like": gamma_check["bounded_0_1_like"],
        "selection_reason": np.nan,
    }


def fit_feature_model(
    df: pd.DataFrame,
    feature: str,
    covariate_cols: list[str],
    covariate_terms: list[str],
    stage1_normality_map: dict | None = None,
) -> dict | None:
    if feature in FORCE_OLS_FEATURES:
        result = fit_ols_hc3(
            df,
            feature,
            covariate_cols=covariate_cols,
            covariate_terms=covariate_terms,
            stage1_normality_map=stage1_normality_map,
        )
        if result is not None:
            result["selection_reason"] = "force_ols_feature"
        return result

    gamma_result = fit_gamma_glm(
        df,
        feature,
        covariate_cols=covariate_cols,
        covariate_terms=covariate_terms,
        stage1_normality_map=stage1_normality_map,
    )

    if gamma_result is not None and gamma_result.get("status") == "ok":
        gamma_result["selection_reason"] = "gamma_checks_passed"
        return gamma_result

    ols_result = fit_ols_hc3(
        df,
        feature,
        covariate_cols=covariate_cols,
        covariate_terms=covariate_terms,
        stage1_normality_map=stage1_normality_map,
    )
    if ols_result is not None:
        if gamma_result is not None:
            ols_result["selection_reason"] = f"fallback_to_ols:{gamma_result.get('gamma_reason', 'unknown')}"
        else:
            ols_result["selection_reason"] = "default_ols"
    return ols_result


 
def main():
    warnings.filterwarnings("ignore")

    stage1_normality_map = load_stage1_normality_map(STAGE1_SUMMARY_CSV)

    df = build_dataframe(DEPRESSION_JSON, NORMAL_JSON)
    df = coerce_numeric(df, FEATURES)

    features = resolve_features(df, FEATURES)
    if not features:
        print("No usable features found.")
        return

    requested_covariates = get_requested_covariate_names(df)
    df, covariate_cols, covariate_terms = prepare_model_covariates(
        df,
        requested_covariates,
        top_n_levels=TOP_N_CONTEXT_LEVELS,
    )

    print(f"Total records: {len(df)}")
    print(f"Group covariate: {GROUP_COVARIATE}")
    print(f"Context setting: {CONTEXT_COVARIATE}")
    print(f"Resolved covariates: {requested_covariates}")
    print(f"Model covariate columns: {covariate_cols}")
    print("Features used:")
    for f in features:
        print("  -", f)

    if stage1_normality_map:
        print(f"\n[Info] Loaded Stage 1 normality summary from: {STAGE1_SUMMARY_CSV.resolve()}")
    else:
        print("\n[Warning] Stage 1 normality summary not available; model selection will rely only on Stage 4 checks.")

    results = []
    skipped_features = []

    for feature in features:
        result = fit_feature_model(
            df,
            feature,
            covariate_cols=covariate_cols,
            covariate_terms=covariate_terms,
            stage1_normality_map=stage1_normality_map,
        )
        if result is not None:
            results.append(result)
        else:
            skipped_features.append(feature)

    if skipped_features:
        print("\n[Info] Skipped features:")
        for f in skipped_features:
            print("  -", f)

    results_df = pd.DataFrame(results)

    if results_df.empty:
        print("No model results produced.")
        return

    ok_mask = (results_df["status"] == "ok") & results_df["p_group"].notna()
    results_df["q_group"] = np.nan
    if ok_mask.sum() > 0:
        _, qvals, _, _ = multipletests(results_df.loc[ok_mask, "p_group"], method="fdr_bh")
        results_df.loc[ok_mask, "q_group"] = qvals

    results_df = results_df.sort_values(
        by=["q_group", "p_group"],
        ascending=[True, True],
        na_position="last"
    ).reset_index(drop=True)

    out_csv = OUTPUT_DIR / "covariate_sensitivity_results.csv"
    results_df.to_csv(
        out_csv,
        index=False,
        encoding="utf-8-sig"
    )

    print("\nDone.")
    print(f"Saved: {out_csv.resolve()}")

    print("\n[Info] Model types used:")
    print(results_df["model_type"].value_counts(dropna=False))

    if "selection_reason" in results_df.columns:
        print("\n[Info] Selection reasons:")
        print(results_df["selection_reason"].value_counts(dropna=False))

    if "status" in results_df.columns:
        print("\n[Info] Status summary:")
        print(results_df["status"].value_counts(dropna=False))


if __name__ == "__main__":
    main()

[Info] 'primary_subreddit' not found; derived from 'subreddits'. If 'subreddits' is a list, the first element is treated as the primary context.
Total records: 2167
Group covariate: group
Context setting: primary_subreddit
Resolved covariates: ['primary_subreddit']
Model covariate columns: ['primary_subreddit__major']
Features used:
  - word_count
  - sentence_count
  - mean_sentence_length
  - semantic_coherence
  - maximum_similarity
  - definite_articles_per_100_words
  - indefinite_articles_per_100_words
  - pronouns_per_100_words
  - first_person_singular_per_100_words
  - first_person_plural_per_100_words
  - second_person_per_100_words
  - third_person_per_100_words
  - noun_overlap
  - pronoun_overlap
  - content_overlap
  - LCC
  - LSC
  - number_of_edges
  - degree_average
  - degree_std
  - LCCr
  - LSCr
  - ambiguous_pronoun_count_per_100_words

[Info] Loaded Stage 1 normality summary from: C:\Users\guoji\Downloads\毕设code\feature_visual_analysis\feature_group_summary.csv

[